# Subgraph Matching

This notebook demonstrates subgraph pattern matching - finding instances of a smaller graph (pattern) within a larger graph (supergraph).

**Adapted from topologicpy to use topologic_fast**

Key concepts:
1. Create a supergraph (larger graph to search in)
2. Create a subgraph pattern (smaller graph to find)
3. Find all matches of the pattern in the supergraph
4. Visualize the results with Plotly

**Note:** topologic_fast doesn't have a built-in `Graph.SubGraphMatches()` function, so we implement a basic subgraph isomorphism algorithm.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from collections import defaultdict
from itertools import permutations

## 1. Create a Supergraph from CellComplex

We'll create a supergraph from a 2x2x2 CellComplex, where:
- Vertices represent cells
- Edges represent cells sharing a face

In [ ]:
# Create a 2x2x2 grid of cells
cells = []
cell_ids = {}  # (i, j, k) -> cell_index
cell_centroids = {}  # cell_index -> (x, y, z)

cell_size = 1.0

idx = 0
for k in range(2):  # z
    for j in range(2):  # y
        for i in range(2):  # x
            x = i * cell_size
            y = j * cell_size
            z = k * cell_size
            
            cell = tf.Cell.Box(x, y, z, cell_size, cell_size, cell_size)
            cells.append(cell)
            
            cell_ids[(i, j, k)] = idx
            cell_centroids[idx] = (x + cell_size/2, y + cell_size/2, z + cell_size/2)
            idx += 1

print(f"Created {len(cells)} cells in a 2x2x2 grid")

In [ ]:
# Create CellComplex
cell_complex = tf.CellComplex.ByCells(cells)

print(f"CellComplex: {cell_complex.NumCells()} cells")

In [ ]:
# Build adjacency based on grid positions (face-sharing neighbors)
def get_neighbors(i, j, k, cell_ids):
    """Get indices of cells adjacent to position (i, j, k)."""
    neighbors = []
    for di, dj, dk in [(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)]:
        pos = (i+di, j+dj, k+dk)
        if pos in cell_ids:
            neighbors.append(cell_ids[pos])
    return neighbors

cell_adjacency = {}
for (i, j, k), idx in cell_ids.items():
    cell_adjacency[idx] = get_neighbors(i, j, k, cell_ids)

print("Cell Adjacency:")
for idx, neighbors in cell_adjacency.items():
    # Find grid position
    for pos, cidx in cell_ids.items():
        if cidx == idx:
            print(f"  Cell {idx} at {pos}: neighbors = {neighbors}")
            break

In [ ]:
# Create supergraph vertices at cell centroids
supergraph_vertices = []
vertex_labels = {}  # vertex index -> label

for idx in range(len(cells)):
    x, y, z = cell_centroids[idx]
    v = tf.Vertex.ByCoordinates(x, y, z)
    supergraph_vertices.append(v)
    vertex_labels[idx] = f"c_{idx+1}"  # Label like c_1, c_2, etc.

print(f"Created {len(supergraph_vertices)} vertices")

In [ ]:
# Create supergraph edges based on adjacency
supergraph_edges = []
edge_set = set()

for idx, neighbors in cell_adjacency.items():
    for neighbor in neighbors:
        edge_key = tuple(sorted([idx, neighbor]))
        if edge_key not in edge_set:
            edge_set.add(edge_key)
            edge = tf.Edge.ByStartVertexEndVertex(
                supergraph_vertices[idx],
                supergraph_vertices[neighbor]
            )
            supergraph_edges.append(edge)

print(f"Created {len(supergraph_edges)} edges")

In [ ]:
# Create supergraph
supergraph = tf.Graph.ByVerticesEdges(supergraph_vertices, supergraph_edges)

print(f"Supergraph Statistics:")
print(f"  Vertices: {supergraph.Order()}")
print(f"  Edges: {supergraph.Size()}")
print(f"  Density: {supergraph.Density():.4f}")
print(f"  Diameter: {supergraph.Diameter()}")

## 2. Create a Subgraph Pattern

We'll create a simple pattern to search for: a path of 3 vertices (triangle without one edge, or linear chain).

In [ ]:
# Create a simple L-shaped pattern (3 vertices, 2 edges)
# This represents 3 cells where cell 1 connects to cell 2, and cell 2 connects to cell 3

pattern_vertices = [
    tf.Vertex.ByCoordinates(0, 0, 0),
    tf.Vertex.ByCoordinates(1, 0, 0),
    tf.Vertex.ByCoordinates(1, 1, 0)
]

pattern_edges = [
    tf.Edge.ByStartVertexEndVertex(pattern_vertices[0], pattern_vertices[1]),
    tf.Edge.ByStartVertexEndVertex(pattern_vertices[1], pattern_vertices[2])
]

subgraph = tf.Graph.ByVerticesEdges(pattern_vertices, pattern_edges)

print(f"Subgraph (Pattern) Statistics:")
print(f"  Vertices: {subgraph.Order()}")
print(f"  Edges: {subgraph.Size()}")
print(f"  Pattern: V0 -- V1 -- V2 (L-shape)")

## 3. Implement Subgraph Matching Algorithm

We implement a basic subgraph isomorphism algorithm to find all occurrences of the pattern in the supergraph.

**Note:** topologic_fast doesn't have `Graph.SubGraphMatches()`, so we implement this manually.

In [ ]:
def get_vertex_key(v):
    """Create hashable key from vertex coordinates."""
    coords = v.Coordinates()
    return (round(coords[0], 4), round(coords[1], 4), round(coords[2], 4))

def build_adjacency_dict(graph):
    """Build adjacency dictionary from graph."""
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Map vertex key to index
    v_to_idx = {}
    idx_to_v = {}
    for i, v in enumerate(vertices):
        key = get_vertex_key(v)
        v_to_idx[key] = i
        idx_to_v[i] = v
    
    # Build adjacency
    adj = defaultdict(set)
    for edge in edges:
        start, end = edge.Vertices()
        start_idx = v_to_idx[get_vertex_key(start)]
        end_idx = v_to_idx[get_vertex_key(end)]
        adj[start_idx].add(end_idx)
        adj[end_idx].add(start_idx)
    
    return adj, v_to_idx, idx_to_v

# Build adjacency for both graphs
super_adj, super_v_to_idx, super_idx_to_v = build_adjacency_dict(supergraph)
sub_adj, sub_v_to_idx, sub_idx_to_v = build_adjacency_dict(subgraph)

print("Supergraph adjacency:")
for idx in sorted(super_adj.keys()):
    print(f"  {idx}: {sorted(super_adj[idx])}")

print("\nSubgraph (pattern) adjacency:")
for idx in sorted(sub_adj.keys()):
    print(f"  {idx}: {sorted(sub_adj[idx])}")

In [ ]:
def is_valid_mapping(mapping, sub_adj):
    """
    Check if a mapping from subgraph vertices to supergraph vertices
    preserves all edges.
    
    mapping: dict mapping subgraph vertex index to supergraph vertex index
    """
    for sub_v, neighbors in sub_adj.items():
        if sub_v not in mapping:
            return False
        super_v = mapping[sub_v]
        
        for neighbor in neighbors:
            if neighbor not in mapping:
                return False
            super_neighbor = mapping[neighbor]
            
            # Check if this edge exists in supergraph
            if super_neighbor not in super_adj[super_v]:
                return False
    
    return True

def find_subgraph_matches(sub_adj, super_adj):
    """
    Find all subgraph isomorphism matches.
    Returns list of mappings (dict: subgraph_idx -> supergraph_idx)
    """
    sub_vertices = list(sub_adj.keys())
    super_vertices = list(super_adj.keys())
    
    n_sub = len(sub_vertices)
    n_super = len(super_vertices)
    
    if n_sub > n_super:
        return []  # Pattern larger than graph
    
    matches = []
    
    # Try all possible mappings of subgraph vertices to supergraph vertices
    # This is brute force - for large graphs, use VF2 algorithm
    for perm in permutations(super_vertices, n_sub):
        mapping = {sub_vertices[i]: perm[i] for i in range(n_sub)}
        
        if is_valid_mapping(mapping, sub_adj):
            # Check if this is a unique match (not a rotation of existing)
            match_set = frozenset(mapping.values())
            is_duplicate = False
            for existing in matches:
                if frozenset(existing.values()) == match_set:
                    is_duplicate = True
                    break
            
            if not is_duplicate:
                matches.append(mapping)
    
    return matches

# Find matches
print("Finding subgraph matches...")
matches = find_subgraph_matches(sub_adj, super_adj)
print(f"Found {len(matches)} matches")

In [ ]:
# Display the matches
print("Subgraph Matches (L-shaped pattern of 3 cells):")
print("=" * 50)

for i, mapping in enumerate(matches):
    # Get supergraph vertex indices in pattern order
    super_indices = [mapping[j] for j in range(len(sub_adj))]
    
    # Get cell labels
    labels = [vertex_labels[idx] for idx in super_indices]
    
    print(f"\nMatch {i+1}:")
    print(f"  Pattern: V0 -- V1 -- V2")
    print(f"  Maps to: {labels[0]} -- {labels[1]} -- {labels[2]}")
    print(f"  Cell indices: {super_indices}")

## 4. Visualize the Supergraph

In [ ]:
def visualize_supergraph(graph, vertex_labels):
    """Visualize the supergraph."""
    fig = go.Figure()
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Draw edges
    for edge in edges:
        start, end = edge.Vertices()
        start_c = start.Coordinates()
        end_c = end.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[start_c[0], end_c[0]],
            y=[start_c[1], end_c[1]],
            z=[start_c[2], end_c[2]],
            mode='lines',
            line=dict(color='gray', width=4),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw vertices
    x = [v.Coordinates()[0] for v in vertices]
    y = [v.Coordinates()[1] for v in vertices]
    z = [v.Coordinates()[2] for v in vertices]
    labels = [vertex_labels[i] for i in range(len(vertices))]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers+text',
        marker=dict(size=20, color='steelblue'),
        text=labels,
        textposition='top center',
        textfont=dict(size=12, color='black'),
        name='Vertices'
    ))
    
    fig.update_layout(
        title='Supergraph (2x2x2 CellComplex dual graph)',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
        ),
        width=800,
        height=600
    )
    
    return fig

fig_super = visualize_supergraph(supergraph, vertex_labels)
fig_super.show()

## 5. Visualize Each Match

In [ ]:
def visualize_match(graph, match, vertex_labels, match_idx, super_idx_to_v):
    """Visualize a specific subgraph match."""
    fig = go.Figure()
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Get matched vertex indices
    matched_indices = set(match.values())
    
    # Build matched edges
    matched_edges = set()
    for sub_v, neighbors in sub_adj.items():
        super_v = match[sub_v]
        for neighbor in neighbors:
            super_neighbor = match[neighbor]
            matched_edges.add(tuple(sorted([super_v, super_neighbor])))
    
    # Draw all edges
    for edge in edges:
        start, end = edge.Vertices()
        start_c = start.Coordinates()
        end_c = end.Coordinates()
        
        # Check if this edge is in the match
        start_key = get_vertex_key(start)
        end_key = get_vertex_key(end)
        start_idx = super_v_to_idx.get(start_key, -1)
        end_idx = super_v_to_idx.get(end_key, -1)
        edge_key = tuple(sorted([start_idx, end_idx]))
        
        if edge_key in matched_edges:
            color = 'blue'
            width = 10
        else:
            color = 'lightgray'
            width = 2
        
        fig.add_trace(go.Scatter3d(
            x=[start_c[0], end_c[0]],
            y=[start_c[1], end_c[1]],
            z=[start_c[2], end_c[2]],
            mode='lines',
            line=dict(color=color, width=width),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw vertices
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        
        if i in matched_indices:
            color = 'blue'
            size = 25
        else:
            color = 'lightgray'
            size = 15
        
        fig.add_trace(go.Scatter3d(
            x=[coords[0]],
            y=[coords[1]],
            z=[coords[2]],
            mode='markers+text',
            marker=dict(size=size, color=color, line=dict(color='black', width=1)),
            text=[vertex_labels[i]],
            textposition='top center',
            textfont=dict(size=10, color='black'),
            showlegend=False
        ))
    
    # Get match description
    super_indices = [match[j] for j in range(len(sub_adj))]
    labels = [vertex_labels[idx] for idx in super_indices]
    
    fig.update_layout(
        title=f'Match {match_idx+1}: {labels[0]} -- {labels[1]} -- {labels[2]}',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
        ),
        width=700,
        height=500
    )
    
    return fig

# Show first few matches
for i, match in enumerate(matches[:6]):  # Show up to 6 matches
    fig_match = visualize_match(supergraph, match, vertex_labels, i, super_idx_to_v)
    fig_match.show()

## 6. Create and Match a Different Pattern

In [ ]:
# Create a triangle pattern (3 vertices, 3 edges - all connected)
triangle_vertices = [
    tf.Vertex.ByCoordinates(0, 0, 0),
    tf.Vertex.ByCoordinates(1, 0, 0),
    tf.Vertex.ByCoordinates(0.5, 1, 0)
]

triangle_edges = [
    tf.Edge.ByStartVertexEndVertex(triangle_vertices[0], triangle_vertices[1]),
    tf.Edge.ByStartVertexEndVertex(triangle_vertices[1], triangle_vertices[2]),
    tf.Edge.ByStartVertexEndVertex(triangle_vertices[2], triangle_vertices[0])
]

triangle_graph = tf.Graph.ByVerticesEdges(triangle_vertices, triangle_edges)

print("Triangle Pattern:")
print(f"  Vertices: {triangle_graph.Order()}")
print(f"  Edges: {triangle_graph.Size()}")
print(f"  Pattern: V0 -- V1 -- V2 -- V0 (triangle)")

In [ ]:
# Build triangle adjacency
tri_adj, tri_v_to_idx, tri_idx_to_v = build_adjacency_dict(triangle_graph)

# Find triangle matches
print("Finding triangle matches...")
triangle_matches = find_subgraph_matches(tri_adj, super_adj)
print(f"Found {len(triangle_matches)} triangle matches")

if len(triangle_matches) == 0:
    print("\nNo triangles found in supergraph.")
    print("This is expected because a 2x2x2 cell grid has no 3-cycles.")
    print("The adjacency graph is bipartite (cells can be 2-colored by parity).")

## 7. Create a Square Pattern

In [ ]:
# Create a square pattern (4 vertices, 4 edges - cycle)
square_vertices = [
    tf.Vertex.ByCoordinates(0, 0, 0),
    tf.Vertex.ByCoordinates(1, 0, 0),
    tf.Vertex.ByCoordinates(1, 1, 0),
    tf.Vertex.ByCoordinates(0, 1, 0)
]

square_edges = [
    tf.Edge.ByStartVertexEndVertex(square_vertices[0], square_vertices[1]),
    tf.Edge.ByStartVertexEndVertex(square_vertices[1], square_vertices[2]),
    tf.Edge.ByStartVertexEndVertex(square_vertices[2], square_vertices[3]),
    tf.Edge.ByStartVertexEndVertex(square_vertices[3], square_vertices[0])
]

square_graph = tf.Graph.ByVerticesEdges(square_vertices, square_edges)

print("Square Pattern (4-cycle):")
print(f"  Vertices: {square_graph.Order()}")
print(f"  Edges: {square_graph.Size()}")

In [ ]:
# Build square adjacency and find matches
sq_adj, sq_v_to_idx, sq_idx_to_v = build_adjacency_dict(square_graph)

print("Finding 4-cycle matches...")
square_matches = find_subgraph_matches(sq_adj, super_adj)
print(f"Found {len(square_matches)} 4-cycle matches")

if square_matches:
    print("\n4-cycle matches (faces of the cube):")
    for i, match in enumerate(square_matches):
        indices = [match[j] for j in range(4)]
        labels = [vertex_labels[idx] for idx in indices]
        print(f"  Match {i+1}: {labels}")

## 8. Graph Comparison (Similarity)

In [ ]:
def compare_graphs(g1, g2):
    """
    Compare two graphs using basic metrics.
    Returns a similarity dictionary.
    
    Note: topologic_fast doesn't have Graph.Compare(), so we implement basic comparison.
    """
    # Basic metrics comparison
    v1 = g1.Order()
    v2 = g2.Order()
    e1 = g1.Size()
    e2 = g2.Size()
    
    # Jaccard similarity for edges
    # (simplified - just comparing edge counts)
    edge_similarity = 1 - abs(e1 - e2) / max(e1, e2, 1)
    
    # Vertex count similarity
    vertex_similarity = 1 - abs(v1 - v2) / max(v1, v2, 1)
    
    # Density similarity
    d1 = g1.Density()
    d2 = g2.Density()
    density_similarity = 1 - abs(d1 - d2)
    
    # Overall similarity
    overall = (edge_similarity + vertex_similarity + density_similarity) / 3
    
    return {
        'vertex_count_similarity': vertex_similarity,
        'edge_count_similarity': edge_similarity,
        'density_similarity': density_similarity,
        'overall_similarity': overall,
        'g1_vertices': v1,
        'g2_vertices': v2,
        'g1_edges': e1,
        'g2_edges': e2
    }

# Compare supergraph with a scaled version of itself
scaled_vertices = []
for v in supergraph_vertices:
    coords = v.Coordinates()
    scaled_v = tf.Vertex.ByCoordinates(coords[0] * 1.1, coords[1] * 1.1, coords[2] * 1.1)
    scaled_vertices.append(scaled_v)

scaled_edges = []
for edge in supergraph_edges:
    start, end = edge.Vertices()
    start_c = start.Coordinates()
    end_c = end.Coordinates()
    
    scaled_start = tf.Vertex.ByCoordinates(start_c[0] * 1.1, start_c[1] * 1.1, start_c[2] * 1.1)
    scaled_end = tf.Vertex.ByCoordinates(end_c[0] * 1.1, end_c[1] * 1.1, end_c[2] * 1.1)
    
    scaled_edges.append(tf.Edge.ByStartVertexEndVertex(scaled_start, scaled_end))

scaled_graph = tf.Graph.ByVerticesEdges(scaled_vertices, scaled_edges)

similarity = compare_graphs(supergraph, scaled_graph)

print("Graph Comparison (supergraph vs scaled supergraph):")
print("=" * 50)
for key, value in similarity.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## Summary

This notebook demonstrated:

1. **Supergraph Creation** - Built a dual graph from a 2x2x2 CellComplex
2. **Pattern Definition** - Created various subgraph patterns (L-shape, triangle, square)
3. **Subgraph Matching** - Implemented isomorphism algorithm to find pattern occurrences
4. **Visualization** - Created 3D visualizations highlighting matches
5. **Graph Comparison** - Basic similarity metrics between graphs

### Key Differences from topologicpy:

- **No SubGraphMatches()**: topologic_fast doesn't have `Graph.SubGraphMatches()`, so we implemented our own
- **No Graph.Compare()**: We implemented basic comparison metrics manually
- **No strict parameter**: We don't have the strict vs non-strict matching option
- **No vertex/edge ID keys**: We use coordinate-based vertex identification

### Applications:
- **Pattern Recognition**: Find recurring spatial patterns in buildings
- **Template Matching**: Verify design rules by finding required patterns
- **Anomaly Detection**: Identify non-conforming structures
- **Graph Analysis**: Structural comparison and similarity assessment